In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from walinet.parameter_calibration.load_data import *

from walinet.parameter_calibration.metab_normalization import *

from walinet.parameter_calibration.compute_statistics import (
    extract_valid_voxels,
    calculate_pooled_median_iqr,
)

from walinet.parameter_calibration.metab_calibration import *

from walinet.parameter_calibration.water_lipid_ratios import *

from walinet.parameter_calibration.pipeline_FWHM_SNR_shifts import (
    calibrate_parameter_from_maps,
)

In [ ]:
bandwidth_hz = 2778.0
nmr_frequency_hz = 297_222_931.0
water_ppm = 4.68

In [ ]:
SUBJECT_DIRS = [
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol03_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol04_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol05_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol07_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol01_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol02_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol03_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol04_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol05_Dat_NoL2_GradDel"
]

In [ ]:
calibration_maps = load_calibration_maps(
    SUBJECT_DIRS,
    suffix=".nii.gz",  # alternativ: ".mnc"
)

for subject_id, subject_data in calibration_maps.items():
    print(
        subject_id,
        len(subject_data["metabolites"]),
        "Metaboliten, FWHM:",
        subject_data["fwhm"].shape,
    )

In [ ]:
TRAIN_CONFIG_PATH = "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/configs/Training/train_7T.yaml"

r_calibration = calculate_r_maps(
    calibration_maps=calibration_maps,
    train_config_path=TRAIN_CONFIG_PATH,
)

In [ ]:
Metabos = [
    "Asp", "Cr", "GABA", "Glc", "Gln", "Glu", "GPC", "GSH",
    "Ins", "NAA", "NAAG", "PCh", "PCr", "Scyllo", "Tau", "TwoHG"
]

for METABO in Metabos:

    calibration = calibrate_metabolite_ratio_from_r_maps(
        r_calibration=r_calibration,
        metabolite_name=f"{METABO}",
        bins=50,
        plot_percentile=99.5,
        truncated_normal_sigma_factor=2.0,
        lognormal_sigma_factor=2.0,
        save_path=f"SavedGraphics/{METABO}_ratio_calibration.pdf",
        show=True,
    )